# Cálculo de Métricas de Desproporcionalidade — Projeto VigiMed  
# Disproportionality Metrics Calculation — VigiMed Project

---

## 📘 Descrição | Description

**PT:**  
Este notebook realiza o cálculo das métricas de desproporcionalidade
utilizadas para detecção de possíveis sinais de segurança de medicamentos.

Ele utiliza a base analítica construída no notebook anterior e calcula
as métricas estatísticas empregadas em farmacovigilância para identificação
de associações medicamento–evento.

**EN:**  
This notebook calculates disproportionality metrics used to detect
potential drug safety signals.

It uses the analytical dataset created in the previous notebook and
computes statistical metrics commonly applied in pharmacovigilance to
identify drug–event associations.

---

## 🔄 Fluxo executado | Workflow

**PT — O notebook executa automaticamente:**

1. Upload da base analítica gerada anteriormente
2. Preparação dos pares medicamento–evento
3. Construção das tabelas de contingência
4. Cálculo das métricas estatísticas
5. Aplicação dos critérios de detecção de sinais
6. Exportação dos resultados finais

**EN — The notebook automatically performs:**

1. Upload of the analytical dataset generated previously
2. Preparation of drug–event pairs
3. Construction of contingency tables
4. Calculation of disproportionality metrics
5. Application of signal detection criteria
6. Export of final results

---

## ▶️ Instruções de uso | Usage instructions

**PT:**

1. Execute primeiro o notebook de construção do banco.
2. Baixe o arquivo `fat_analitica.parquet` gerado.
3. Faça upload do arquivo quando solicitado neste notebook.
4. Execute todas as células.

**EN:**

1. Run the database construction notebook first.
2. Download the generated `fat_analitica.parquet` file.
3. Upload the file when requested in this notebook.
4. Run all notebook cells.

---
## 📥 Entrada necessária | Required input

**PT:**  
Arquivo gerado pelo notebook anterior:

**EN:**  
File generated by the previous notebook:


fat_analitica.parquet


---


## 📤 Resultado gerado | Output generated

**PT:**  
Arquivo contendo as métricas calculadas e os sinais detectados:

**EN:**  
File containing calculated metrics and detected safety signals:


sinais_detectados.parquet



---

## 📧 Contato | Contact

Ana Carolina Jacoby  
Email: anacarolinajacoby0@gmail.com



## 1- Instalação das dependências | Installing dependencies

**PT:**  
Esta etapa instala automaticamente os programas necessários para executar
o notebook.

**EN:**  
This step automatically installs the required software packages needed to
run the notebook.


In [ ]:
print("Instalando dependências...")

!pip install duckdb pandas pyarrow --quiet

print("Dependências instaladas com sucesso.")


## 2- Importação das bibliotecas | Importing libraries

**PT:**  
Carrega as bibliotecas utilizadas para manipulação dos dados e cálculo das
métricas estatísticas.

**EN:**  
Loads the libraries used for data manipulation and statistical calculations.


In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import os

print("Bibliotecas carregadas com sucesso.")


## 2.1- Registro do ambiente de execução | Execution environment record

**PT:**  
Esta etapa registra as versões das bibliotecas utilizadas para permitir a
reprodução futura dos resultados.

**EN:**  
This step records the versions of libraries used to allow future
reproducibility of the results.


In [ ]:
import sys
import duckdb
import pandas as pd
import numpy as np

print("Versões do ambiente:")
print("Python:", sys.version)
print("duckdb:", duckdb.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 3- Upload da base analítica | Upload analytical dataset

**PT:**  
Faça upload do arquivo `fat_analitica.parquet`, gerado no notebook de
construção do banco.

Este arquivo será utilizado para cálculo das métricas.

**EN:**  
Upload the file `fat_analitica.parquet`, generated in the database
construction notebook.

This file will be used to compute disproportionality metrics.


In [ ]:
from google.colab import files

print("Selecione o arquivo fat_analitica.parquet para upload:")

uploaded = files.upload()

print("Upload concluído.")


## 4- Verificação do arquivo enviado | Uploaded file check

**PT:**  
Esta etapa verifica se o arquivo necessário foi enviado corretamente.

Caso o arquivo esteja ausente ou com nome diferente, a execução será
interrompida para evitar erros posteriores.

**EN:**  
This step checks whether the required file was correctly uploaded.

If the file is missing or has a different name, execution stops to avoid
later errors.


In [ ]:
parquet_files = [f for f in uploaded.keys() if f.endswith(".parquet")]

if len(parquet_files) == 0:
    raise ValueError("Nenhum arquivo .parquet foi enviado.")

if len(parquet_files) > 1:
    raise ValueError("Envie apenas um arquivo .parquet por vez.")

required_file = parquet_files[0]

print(f"Arquivo identificado automaticamente: {required_file}")

df = pd.read_parquet(required_file)

print("Base carregada com sucesso.")
print("Dimensão da base:", df.shape)




In [ ]:
df.head()

## 6- Construção das tabelas de contingência | Contingency table construction

**PT:**  
Nesta etapa são calculadas as contagens necessárias para construir as
tabelas de contingência utilizadas nos cálculos de desproporcionalidade.

Para cada par medicamento–evento são obtidas as contagens:

a: medicamento + evento  
b: medicamento + outros eventos  
c: outros medicamentos + evento  
d: outros medicamentos + outros eventos  

**EN:**  
In this step, counts required to build contingency tables used in
disproportionality calculations are computed.

For each drug–event pair, the following counts are obtained:

a: drug + event  
b: drug + other events  
c: other drugs + event  
d: other drugs + other events




## 6.1 - Carregamento em pandas para facilitar a construção

In [ ]:
print("Preparando base para cálculo da matriz...")
print("Preparing dataset for contingency matrix...")

df_matrix_base = (
    df[["ID", "PRINCIPIO_ATIVO", "REACAO_PT", "ATC_CODE_LEVEL_4"]]
    .drop_duplicates()
)

print("Dimensão da base para cálculo:", df_matrix_base.shape)


## 6.2 Construção da matriz

In [ ]:
def build_matrix(df):
    # 1. Garantir que o ATC esteja na base de trabalho
    x = df.drop_duplicates(["ID", "PRINCIPIO_ATIVO", "REACAO_PT"])

    # 2. Totais globais (Denominadores permanecem por fármaco e evento)
    n_total = x["ID"].nunique()
    n_drug = x.groupby("PRINCIPIO_ATIVO")["ID"].nunique()
    n_event = x.groupby("REACAO_PT")["ID"].nunique()

    # 3. Interseção (a): Incluímos o ATC aqui para ele "sobreviver" ao agrupamento
    pairs = x.groupby(
        ["PRINCIPIO_ATIVO", "ATC_CODE_LEVEL_4", "REACAO_PT"]
    )["ID"].nunique()

    df_out = pairs.reset_index(name="a")

    # 4. Cálculos (b, c, d) - Continuam usando PRINCIPIO_ATIVO e REACAO_PT
    df_out["b"] = df_out["PRINCIPIO_ATIVO"].map(n_drug) - df_out["a"]
    df_out["c"] = df_out["REACAO_PT"].map(n_event) - df_out["a"]
    df_out["d"] = n_total - df_out["a"] - df_out["b"] - df_out["c"]

    # --- CORREÇÃO DE HALDANE-ANSCOMBE ---
    df_out["a_adj"] = df_out["a"].astype(float)
    df_out["b_adj"] = df_out["b"].astype(float)
    df_out["c_adj"] = df_out["c"].astype(float)
    df_out["d_adj"] = df_out["d"].astype(float)

    mask_zero = (df_out['a'] == 0) | (df_out['b'] == 0) | (df_out['c'] == 0) | (df_out['d'] == 0)
    df_out.loc[mask_zero, ['a_adj', 'b_adj', 'c_adj', 'd_adj']] += 0.5

    df_out["n_total"] = n_total

    return df_out


In [ ]:
print("Construindo matriz de contingência...")
print("Building contingency matrix...")

df_matrix = build_matrix(df_matrix_base)

print("Matriz construída.")
print("Contingency matrix constructed.")

print(f"Número de pares medicamento–evento: {len(df_matrix):,}")
print(f"Número total de notificações: {df_matrix.n_total.iloc[0]:,}")


In [ ]:
check = (df_matrix["a"] + df_matrix["b"] + df_matrix["c"] + df_matrix["d"]) == df_matrix["n_total"]
print("Todas as linhas satisfazem a+b+c+d = N ?", check.all())


## 7- Cálculo da Razão de Chances de Notificação (ROR) | Reporting Odds Ratio Calculation

**PT:**  
A Razão de Chances de Notificação (Reporting Odds Ratio – ROR) é uma das
métricas mais utilizadas em farmacovigilância para detectar sinais de
notificação desproporcional de reações adversas a medicamentos.

Ela compara a chance de um evento adverso ser notificado para um
determinado medicamento com a chance de notificação do mesmo evento para
todos os demais medicamentos na base de dados.

Essa métrica auxilia na identificação de potenciais problemas de segurança,
mas **não estabelece causalidade**, apenas sinaliza associações que merecem
investigação adicional.

### Interpretação

- **ROR = 1**: Não há desproporcionalidade na notificação.
- **ROR > 1**: Indica maior frequência relativa de notificação,
  sugerindo possível sinal de segurança.
- **ROR < 1**: Indica menor frequência relativa de notificação.

### Intervalo de confiança (95%)

Para que um ROR seja considerado estatisticamente significativo,
o **limite inferior do intervalo de confiança de 95% deve ser maior que 1**,
indicando que a associação observada provavelmente não ocorreu ao acaso.

---

**EN:**  
The Reporting Odds Ratio (ROR) is one of the most commonly used metrics in
pharmacovigilance to detect signals of disproportionate reporting of adverse
drug reactions.

It compares the odds of reporting a specific adverse event for a given drug
with the odds of reporting the same event for all other drugs in the
database.

This metric helps identify potential safety concerns but **does not imply
causality**, only statistical association requiring further investigation.

### Interpretation

- **ROR = 1**: No disproportionality in reporting.
- **ROR > 1**: Indicates increased reporting frequency, suggesting a
  potential safety signal.
- **ROR < 1**: Indicates lower reporting frequency.

### 95% Confidence Interval

An ROR is considered statistically significant when the **lower bound of
the 95% confidence interval is greater than 1**, suggesting the association
is unlikely due to chance.


In [ ]:
def calculate_ror(a, b, c, d):
    # Aplica a correção APENAS se houver zero em alguma célula
    if a == 0 or b == 0 or c == 0 or d == 0:
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5

    # Cálculo do ROR
    ror = (a * d) / (b * c)

    # Erro Padrão do Log(ROR)
    se_log = np.sqrt(1/a + 1/b + 1/c + 1/d)

    # Intervalo de Confiança 95%
    ci_low = np.exp(np.log(ror) - 1.96 * se_log)
    ci_high = np.exp(np.log(ror) + 1.96 * se_log)

    return ror, ci_low, ci_high

### 7.1- Aplicação do cálculo do ROR | Applying ROR calculation

**PT:**  
Nesta etapa aplicamos a função de cálculo do ROR a cada par
medicamento–evento presente na matriz de contingência.

O resultado é adicionado como novas colunas contendo o valor do ROR e seu
intervalo de confiança.

**EN:**  
In this step, the ROR calculation function is applied to each drug–event
pair in the contingency matrix.

The resulting ROR values and confidence intervals are added as new columns.


In [ ]:
print("Calculando ROR para todos os pares...")

# Identificar onde a correção é necessária (onde há zeros)
mask = (df_matrix['a'] == 0) | (df_matrix['b'] == 0) | (df_matrix['c'] == 0) | (df_matrix['d'] == 0)

# Criar variáveis temporárias para o cálculo (sem alterar os originais de df_matrix)
a = df_matrix['a'].copy()
b = df_matrix['b'].copy()
c = df_matrix['c'].copy()
d = df_matrix['d'].copy()

# Aplicar a correção apenas na máscara
a[mask] += 0.5
b[mask] += 0.5
c[mask] += 0.5
d[mask] += 0.5

# Cálculos vetorizados (muito mais rápidos que .apply)
df_matrix['ROR'] = (a * d) / (b * c)
se_log = np.sqrt(1/a + 1/b + 1/c + 1/d)

df_matrix['ROR_Lower_CI'] = np.exp(np.log(df_matrix['ROR']) - 1.96 * se_log)
df_matrix['ROR_Upper_CI'] = np.exp(np.log(df_matrix['ROR']) + 1.96 * se_log)

print("ROR calculado com sucesso (via vetorização).")


## 8- Cálculo do PRR | PRR Calculation

**PT:**  
Nesta etapa é definida a função para cálculo do Proportional Reporting Ratio
(PRR) e seu intervalo de confiança para pares medicamento–evento.

O PRR mede se um evento adverso é relatado proporcionalmente com maior
frequência para um medicamento em comparação com todos os demais
medicamentos na base de dados.

Esta métrica indica **desproporcionalidade de notificação**, mas não
estabelece causalidade.

### Interpretação

- **PRR = 1**: ausência de associação.
- **PRR > 1**: maior frequência relativa do evento para o medicamento.
- **PRR < 1**: menor frequência relativa.

### Intervalo de confiança (95%)

O PRR é considerado estatisticamente significativo quando o limite inferior
do intervalo de confiança é maior ou igual a 1.

Critérios adicionais de sinal (ex.: número mínimo de casos e valor de
Qui-quadrado) são aplicados posteriormente.

---

**EN:**  
This step defines the function used to calculate the Proportional Reporting
Ratio (PRR) and its confidence interval for drug–event pairs.

PRR measures whether an adverse event is reported proportionally more often
for a drug compared to all other drugs in the database.

It indicates reporting disproportionality but does not establish causality.

### Interpretation

- **PRR = 1**: no association.
- **PRR > 1**: increased reporting frequency.
- **PRR < 1**: decreased reporting frequency.

### 95% Confidence Interval

PRR is considered statistically significant when the lower bound of the
confidence interval is greater than or equal to 1.

Additional signal criteria are applied later.


In [ ]:
def calculate_prr(a, b, c, d):
    """
    Calcula PRR, intervalo de confiança e qui-quadrado com correção condicional.
    """
    #  Tratamento de divisões por zero antes da correção
    if (a + b == 0) or (c + d == 0):
        return np.nan, np.nan, np.nan, np.nan

    #  Aplicação condicional da Correção de Haldane-Anscombe
    # Se houver zero em qualquer célula da tabela 2x2
    if a == 0 or b == 0 or c == 0 or d == 0:
        a_c, b_c, c_c, d_c = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    else:
        a_c, b_c, c_c, d_c = a, b, c, d

    # 3. Cálculo do PRR pontual (usando valores ajustados se necessário)
    p1 = a_c / (a_c + b_c)
    p2 = c_c / (c_c + d_c)

    # Prevenção extra para p2 zero (mesmo com correção, para segurança do código)
    if p2 == 0:
        return np.nan, np.nan, np.nan, np.nan

    prr = p1 / p2

    #  Cálculo do Erro Padrão e Intervalo de Confiança
    # A fórmula do SE para PRR é baseada na variância do logaritmo da proporção
    se_log = np.sqrt((1/a_c) - (1/(a_c + b_c)) + (1/c_c) - (1/(c_c + d_c)))

    ci_low = np.exp(np.log(prr) - 1.96 * se_log)
    ci_high = np.exp(np.log(prr) + 1.96 * se_log)

    #  Qui-quadrado (Yates' correction recomendada para farmacovigilância)
    # A literatura muitas vezes prefere o Qui-quadrado com correção de Yates
    # quando as frequências são baixas.
    N = a + b + c + d
    denom = (a + b) * (c + d) * (a + c) * (b + d)

    if denom == 0:
        chi2 = np.nan
    else:
        # Fórmula com correção de Yates para maior rigor
        chi2 = (N * (max(0, abs(a * d - b * c) - (N / 2))**2)) / denom

    return prr, ci_low, ci_high, chi2


### 8.1- Aplicação do cálculo do PRR | Applying PRR calculation

**PT:**  
Nesta etapa aplicamos a função de cálculo do PRR a cada par
medicamento–evento presente na matriz de contingência.

O resultado inclui o valor do PRR, seu intervalo de confiança e o valor
do teste Qui-quadrado.

**EN:**  
In this step, the PRR calculation function is applied to each drug–event
pair in the contingency matrix.

The result includes PRR values, confidence intervals and the Chi-square
statistic.


In [ ]:
print("Calculando PRR para todos os pares...")

# Identificar onde a correção é necessária (onde há zeros)
mask = (df_matrix['a'] == 0) | (df_matrix['b'] == 0) | (df_matrix['c'] == 0) | (df_matrix['d'] == 0)

# Criar variáveis temporárias para o cálculo
a = df_matrix['a'].copy()
b = df_matrix['b'].copy()
c = df_matrix['c'].copy()
d = df_matrix['d'].copy()

# Aplicar a correção de Haldane-Anscombe apenas na máscara
a[mask] += 0.5
b[mask] += 0.5
c[mask] += 0.5
d[mask] += 0.5

# Cálculo do PRR (vetorizado)
p1 = a / (a + b)
p2 = c / (c + d)
df_matrix['PRR'] = p1 / p2

# Cálculo do Erro Padrão e Intervalos de Confiança
se_log_prr = np.sqrt((1/a) - (1/(a + b)) + (1/c) - (1/(c + d)))
df_matrix['PRR_Lower_CI'] = np.exp(np.log(df_matrix['PRR']) - 1.96 * se_log_prr)
df_matrix['PRR_Upper_CI'] = np.exp(np.log(df_matrix['PRR']) + 1.96 * se_log_prr)

# Cálculo do Qui-quadrado com Correção de Yates
N = df_matrix['a'] + df_matrix['b'] + df_matrix['c'] + df_matrix['d']
num = N * (np.maximum(0, np.abs(df_matrix['a']*df_matrix['d'] - df_matrix['b']*df_matrix['c']) - (N/2))**2)
den = (df_matrix['a']+df_matrix['b']) * (df_matrix['c']+df_matrix['d']) * (df_matrix['a']+df_matrix['c']) * (df_matrix['b']+df_matrix['d'])

# Evitar divisão por zero no Qui-quadrado
df_matrix['Chi_Square'] = np.where(den != 0, num / den, np.nan)

print("PRR e Qui-quadrado calculados com sucesso.")


In [ ]:
df_matrix[["PRINCIPIO_ATIVO", "REACAO_PT", "a", "ROR", "PRR"]].head()


## 9-  Medidas Bayesianas de Desproporcionalidade | Bayesian Disproportionality Measures

**PT:**  
Nesta etapa são implementadas métricas baseadas em inferência Bayesiana,
em particular o Componente de Informação (IC), utilizado pelo algoritmo
BCPNN da Organização Mundial da Saúde.

Diferentemente das métricas frequentistas, abordagens Bayesianas aplicam
mecanismos de "encolhimento" (shrinkage), tornando estimativas mais
estáveis quando o número de notificações é pequeno e reduzindo sinais
falsos positivos.

---

**EN:**  
In this step, Bayesian disproportionality measures are implemented,
specifically the Information Component (IC) used by the WHO BCPNN
algorithm.

Unlike frequentist metrics, Bayesian approaches apply shrinkage mechanisms,
increasing stability in small-count situations and reducing false positive
signals.


### 9.1 Componente de Informação (IC) | Information Component (IC)

**PT:**  
O IC compara a probabilidade observada de um par medicamento–evento com a
probabilidade esperada na base de dados, utilizando logaritmo base 2.

Valores positivos indicam que o evento ocorre mais frequentemente do que o
esperado.

Um possível sinal é considerado quando o limite inferior do intervalo de
credibilidade é maior que zero.

---

**EN:**  
The IC compares the observed probability of a drug–event pair with the
expected probability in the database using base-2 logarithm.

Positive values indicate higher-than-expected reporting frequency.

A potential signal occurs when the lower bound of the credibility interval
is greater than zero.


In [ ]:
print("Calculando IC (Information Component) para todos os pares...")

#  Variáveis auxiliares
n = df_matrix['a'] + df_matrix['b'] + df_matrix['c'] + df_matrix['d']
# Valor esperado: ((a+b)*(a+c)) / n
expected = ((df_matrix['a'] + df_matrix['b']) * (df_matrix['a'] + df_matrix['c'])) / n

# Cálculo do IC com a correção bayesiana (suavização)
# O padrão do UMC é adicionar 0.5 tanto ao observado quanto ao esperado
df_matrix['IC'] = np.log2((df_matrix['a'] + 0.5) / (expected + 0.5))

# Cálculo do Erro Padrão (Desvio Padrão da distribuição posterior)
# Usamos a fórmula aproximada para o intervalo de credibilidade de 95%
numerador = np.maximum(n - df_matrix['a'], 0)
denominador = (df_matrix['a'] + 0.5) * (n + 1)
se_ic = (1 / np.log(2)) * np.sqrt(numerador / denominador)

# Limites do Intervalo de Credibilidade (95%)
df_matrix['IC_Lower_CI'] = df_matrix['IC'] - 1.96 * se_ic
df_matrix['IC_Upper_CI'] = df_matrix['IC'] + 1.96 * se_ic

print("IC calculado com sucesso.")


### 9.1.1- Aplicação do cálculo do IC | Applying IC calculation

**PT:**  
Nesta etapa aplicamos o cálculo do Componente de Informação (IC) a cada
par medicamento–evento presente na matriz de contingência.

O resultado inclui o valor do IC e seus limites inferior e superior do
intervalo de credibilidade.

**EN:**  
In this step, the Information Component (IC) calculation is applied to each
drug–event pair in the contingency matrix.

The result includes IC values and the lower and upper bounds of the
credibility interval.


In [ ]:
print("Calculando IC para todos os pares...")

# Variáveis auxiliares para o cálculo
n = df_matrix['a'] + df_matrix['b'] + df_matrix['c'] + df_matrix['d']
# Valor esperado (E): ((a+b) * (a+c)) / n
expected = ((df_matrix['a'] + df_matrix['b']) * (df_matrix['a'] + df_matrix['c'])) / n

# Cálculo do IC com suavização Bayesiana (+0.5)
# O padrão do UMC é adicionar 0.5 ao observado e ao esperado
df_matrix['IC_BCPNN'] = np.log2((df_matrix['a'] + 0.5) / (expected + 0.5))

# Cálculo do Erro Padrão (Desvio Padrão da distribuição posterior)
# Usamos a aproximação para o intervalo de credibilidade de 95%
numerador = np.maximum(n - df_matrix['a'], 0)
denominador = (df_matrix['a'] + 0.5) * (n + 1)
se_ic = (1 / np.log(2)) * np.sqrt(numerador / denominador)

# Limites do Intervalo de Credibilidade (95%)
df_matrix['IC_Lower_CI'] = df_matrix['IC_BCPNN'] - 1.96 * se_ic
df_matrix['IC_Upper_CI'] = df_matrix['IC_BCPNN'] + 1.96 * se_ic

print("IC calculado com sucesso via vetorização.")

### 10- Média Geométrica Bayesiana Empírica (EBGM) | Empirical Bayes Geometric Mean (EBGM)

**PT:**  
O EBGM é uma medida baseada no modelo Gamma-Poisson utilizada para ajustar
estimativas de desproporcionalidade, reduzindo a instabilidade causada por
pequenos números de notificações.

O valor EBGM representa uma estimativa estabilizada da razão entre o número
de relatos observados e esperados.

### Interpretação

- **EBGM ≈ 1**: ocorrência compatível com o esperado.
- **EBGM > 1**: evento relatado com maior frequência que o esperado.

O limite inferior do intervalo de credibilidade de 90%, denominado EB05,
é frequentemente utilizado como critério de sinal.

---

**EN:**  
EBGM is based on the Gamma-Poisson model and provides a stabilized estimate
of disproportionality, reducing variability caused by small counts.

EBGM represents a shrinkage-adjusted ratio between observed and expected
reports.

### Interpretation

- **EBGM ≈ 1**: reporting frequency consistent with expectation.
- **EBGM > 1**: higher-than-expected reporting.

The lower bound of the 90% credibility interval, called EB05, is commonly
used for signal detection.


In [ ]:
print("Calculando EBGM para todos os pares...")

# Variáveis auxiliares
n = df_matrix['a'] + df_matrix['b'] + df_matrix['c'] + df_matrix['d']
expected = ((df_matrix['a'] + df_matrix['b']) * (df_matrix['a'] + df_matrix['c'])) / n

# Cálculo do EBGM (com suavização de 0.5)
# O EBGM "puxa" o resultado para 1 quando os dados são escassos
df_matrix['EBGM'] = (df_matrix['a'] + 0.5) / (expected + 0.5)

# Cálculo do Erro Padrão (aproximação log-normal)
se_ebgm = np.sqrt(1/(df_matrix['a'] + 0.5) + 1/(expected + 0.5))

# Limites do Intervalo de Credibilidade de 90% (EB05 e EB95)
# Usamos 1.645 para 90% de confiança, conforme o padrão do EBGM/MGPS
df_matrix['EB05'] = df_matrix['EBGM'] * np.exp(-1.645 * se_ebgm)
df_matrix['EB95'] = df_matrix['EBGM'] * np.exp(1.645 * se_ebgm)

print("EBGM calculado com sucesso via vetorização.")


### 10.1- Aplicação do cálculo do EBGM | Applying EBGM calculation

**PT:**  
Nesta etapa aplicamos o cálculo do EBGM a cada par medicamento–evento da
matriz de contingência.

**EN:**  
In this step, EBGM is calculated for each drug–event pair in the
contingency matrix.


In [ ]:
print("Calculando EBGM para todos os pares...")


n = df_matrix['a'] + df_matrix['b'] + df_matrix['c'] + df_matrix['d']
expected = ((df_matrix['a'] + df_matrix['b']) * (df_matrix['a'] + df_matrix['c'])) / n

# Cálculo do EBGM com suavização (+0.5)
# O EBGM é a razão entre o observado e o esperado, ajustado por um prior Bayesiano
df_matrix['EBGM_MGPS'] = (df_matrix['a'] + 0.5) / (expected + 0.5)

# Cálculo do Erro Padrão (aproximação log-normal)
se_ebgm = np.sqrt(1/(df_matrix['a'] + 0.5) + 1/(expected + 0.5))

#Cálculo dos limites do intervalo de credibilidade de 90% (EB05 e EB95)
# Usamos 1.645 para 90% de confiança, seguindo o padrão do algoritmo MGPS
df_matrix['EBGM_E05'] = df_matrix['EBGM_MGPS'] * np.exp(-1.645 * se_ebgm)
df_matrix['EBGM_E95'] = df_matrix['EBGM_MGPS'] * np.exp(1.645 * se_ebgm)

print("EBGM calculado com sucesso via vetorização.")


## Conferência do DF

In [ ]:
# Mostra as primeiras 10 linhas
display(df_matrix.head(10))

# Mostra informações sobre as colunas e se há valores nulos
print(df_matrix.info())

## 11- Exportação e download dos resultados | Export and download results

**PT:**  
O dataset completo é salvo e disponibilizado para download.  
Após a execução, o navegador iniciará automaticamente o download do arquivo.

**EN:**  
The full dataset is saved and made available for download.  
After execution, your browser will automatically download the file.


In [ ]:
from google.colab import files

print("Salvando resultados...")
print("Saving results...")

# Exportação em Parquet (uso analítico)
df_matrix.to_parquet("metricas_completas.parquet", index=False)

# Exportação em CSV (uso em Excel e leitura simples)
df_matrix.to_csv("metricas_completas.csv", index=False)

print("Arquivos gerados com sucesso.")
print("Files successfully generated.")
print("Iniciando downloads...")
print("Starting downloads...")

files.download("metricas_completas.parquet")
files.download("metricas_completas.csv")



## Processo concluído | Process completed


In [ ]:
print("""
Processo concluído com sucesso!

Todas as métricas foram calculadas e o dataset final foi exportado.

Agora você pode aplicar filtros específicos ou seguir para etapas
de análise e visualização.
""")
